<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 40px; border-radius: 12px; border: 1px solid #30363d; text-align: center; color: white;">
  <span style="background: rgba(255,255,255,0.2); border: 1px solid rgba(255,255,255,0.4); color: white; padding: 4px 14px; border-radius: 20px; font-size: 12px; font-weight: 600; text-transform: uppercase;">Kafka Training · Lab 12</span>
  <h1 style="color: #ffffff; font-size: 2.4em; font-weight: bold; margin-top: 15px;">Kafka Streams KStream-KTable Join</h1>
  <p style="color: #e0e0e0; font-size: 1.1em;">Learn how to join an event stream with a reference table for data enrichment.</p>
</div>

---

## 🎯 Overview

In this lab, we'll build a Java Kafka Streams application that enriches a stream of clicks with user reference data.

**We will create:**
- `users-topic` (KTable): Contains static reference data (e.g. User ID -> Region)
- `clicks-topic` (KStream): Contains real-time events (e.g. User ID -> URL Clicked)
- `enriched-clicks-topic` (Output): The joined and filtered result.

---


## <span style="color: #667eea;">Step 1:</span> Ensure Kafka Cluster is Running


In [ ]:
!docker-compose -f ../../docker-compose.yml up -d
import time
time.sleep(5)

## <span style="color: #667eea;">Step 2:</span> Create Topics

Notice we configure `users-topic` with log compaction since it acts as a KTable (reference data).

In [ ]:
!docker exec kafka kafka-topics --bootstrap-server localhost:9092 --create --topic users-topic --partitions 1 --replication-factor 1 --config cleanup.policy=compact --if-not-exists
!docker exec kafka kafka-topics --bootstrap-server localhost:9092 --create --topic clicks-topic --partitions 1 --replication-factor 1 --if-not-exists
!docker exec kafka kafka-topics --bootstrap-server localhost:9092 --create --topic enriched-clicks-topic --partitions 1 --replication-factor 1 --if-not-exists
print("\n✓ Topics created successfully!")

## <span style="color: #667eea;">Step 3:</span> Produce Reference Data to the KTable

First, we populate our `users-topic` so our KTable has data to join against.

In [ ]:
from confluent_kafka import Producer

producer = Producer({'bootstrap.servers': 'localhost:9092'})
users = {
    "user1": "US",
    "user2": "EU",
    "user3": "US"
}

print("👤 Producing user reference data...")
for user_id, region in users.items():
    producer.produce('users-topic', key=user_id.encode('utf-8'), value=region.encode('utf-8'))
    print(f"  ✓ {user_id} -> {region}")
    
producer.flush()
print("\n✅ Reference data loaded!")

## <span style="color: #667eea;">Step 4:</span> Compile and Run the Java Streams Application

The application performs the join. We will start it now so it's ready to process the clicks.

In [ ]:
import os
import subprocess
import time

# Ensure we are in the Lab12 root directory
if os.path.basename(os.getcwd()) == 'stream-join-app':
    os.chdir('..')

print("🔨 Compiling Java project with Maven...")
!mvn -f stream-join-app/pom.xml clean compile

print("\n🚀 Starting StreamJoinProcessor in the background...")
process = subprocess.Popen(
    "mvn exec:java -Dexec.mainClass=com.example.StreamJoinProcessor", 
    cwd="stream-join-app",
    shell=True
)

print("⏳ Waiting 15 seconds for it to start up, load the KTable, and wait for streams...")
time.sleep(15)
print("✅ App is running!")

## <span style="color: #667eea;">Step 5:</span> Produce Event Data to the KStream

Now, simulate real-time clicks. Our stream processor will join these on the fly.

In [ ]:
clicks = [
    ("user1", "/home"),
    ("user2", "/pricing"), # Should be filtered out (EU)
    ("user3", "/cart")
]

print("💻 Producing click events...")
for user_id, url in clicks:
    producer.produce('clicks-topic', key=user_id.encode('utf-8'), value=url.encode('utf-8'))
    print(f"  ✓ {user_id} clicked {url}")
    
producer.flush()
print("\n⏳ Waiting 5 seconds for processing...")
import time; time.sleep(5)
print("\n🛑 Stopping stream processor...")
process.terminate()
process.wait()
print("✅ Done!")

## <span style="color: #667eea;">Step 6:</span> Verify the Enriched Output

You should see ONLY the enriched US clicks in the output topic!

In [ ]:
print("🔍 Consuming from enriched-clicks-topic:\n")
!docker exec kafka kafka-console-consumer --bootstrap-server localhost:9092 --topic enriched-clicks-topic --from-beginning --max-messages 2 --property print.key=true --property key.separator=" | " --timeout-ms 5000
print("\n✅ Lab Complete!")